# Market Mayhem: Unified Compendium ML Pipeline

This notebook demonstrates the continuous learning pipeline using the `market_mayhem_compendium.jsonl` artifact. It loads the structured High-Dimensional Knowledge Graph (HDKG) data and human-in-the-loop analyst labels to refine the internal risk probability maps and credit analysis capability modules.

In [ ]:
import json
import pandas as pd
import numpy as np

# Load the JSONL compendium
data_path = '../data/market_mayhem_compendium.jsonl'
records = []

with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

In [ ]:
# Flatten the nested HDKG schema for model training
flattened_data = []

for r in records:
    if 'v30.1_knowledge_graph' in r:
        kg = r['v30.1_knowledge_graph']
        labels = r.get('ml_labels', {})
        
        flat_row = {
            'timestamp': r['analysis_timestamp_utc'],
            'thought_signature': r['thought_signature'],
            'target_ticker': kg['meta_identification_node']['target_ticker'],
            'leverage_ratio': kg['apex_credit_floor']['leverage_ratio'],
            'fcf_to_debt_ratio': kg['apex_credit_floor']['fcf_to_debt_ratio'],
            'intrinsic_value': kg['valuation_matrix']['intrinsic_value'],
            'wacc': kg['valuation_matrix']['wacc'],
            'trend_velocity': kg['liquid_dynamics']['trend_velocity'],
            'black_swan_amplitude': kg['liquid_dynamics']['black_swan_amplitude'],
            'system_conviction_score': kg['strategic_synthesis']['conviction_score'],
            'target_analyst_rating': labels.get('analyst_confirmed_rating', 'NR'),
            'target_analyst_confidence': labels.get('analyst_confidence_score_on_rating_percent', 0)
        }
        flattened_data.append(flat_row)

df = pd.DataFrame(flattened_data)
display(df.head())

### Training the Credit Analysis Capability Module (CACM)
We use the deterministic `apex_credit_floor` metrics and `liquid_dynamics` telemetry to predict the `target_analyst_rating`.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# Feature selection
features = ['leverage_ratio', 'fcf_to_debt_ratio', 'wacc', 'black_swan_amplitude']
X = df[features]
y = df['target_analyst_rating']

# Encode categorical labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train simple model (Proof of Concept)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y_encoded)

print("Model trained successfully. Ready for inference pipelines.")